
# RAG: Retrieval-Augmented Generation

RAG is a technique that enhances the responses generated by an LLM by retrieving relevant information from external knowledge sources and providing that information as additional context to the LLM before generating a response.

It typically involves the following steps:

- **Chunking:** Dividing documents into smaller, manageable pieces.
- **Embedding:** Converting text chunks into numerical vector representations.
- **Indexing:** Storing embeddings in a vector database or search index.
- **Retrieval:** Fetching relevant chunks based on the user's query.
- **Context Augmentation:** Providing the retrieved information to the LLM as additional context.
- **Generation:** Generating a response using the retrieved context.

---

## Why Do We Need RAG?

RAG helps overcome some of the limitations of LLMs, such as:

1. **Outdated Knowledge:** LLMs may not have access to the latest information.
2. **Hallucinations:** LLMs can sometimes generate incorrect or fabricated information.
3. **Lack of Domain-Specific Information:** LLMs may not have access to private or organization-specific knowledge.

### Benefits of RAG

RAG helps generate more accurate, up-to-date, and contextually relevant responses by providing external knowledge to the LLM without requiring retraining or fine-tuning whenever the knowledge base changes.

### Key Takeaway

**RAG = External Knowledge + Retrieval + LLM Generation**

RAG enhances LLM responses by providing relevant external context at inference time without modifying the underlying model weights.


In [ ]:
!pip install -q qdrant-client sentence-transformers requests groq
!pip install -q requests pypdf

#### **Step 1. Load PDF document**

In [ ]:
import os
import requests
from io import BytesIO
from pypdf import PdfReader

pdf_url = "https://raw.githubusercontent.com/janvi20/test-data-hub/main/RAG/student_financial_aid_scholarship_policy_rag.pdf"

def load_document(url: str) -> str:
    response = requests.get(url, timeout=10)
    response.raise_for_status()

    # Read PDF binary content
    pdf_file = BytesIO(response.content)

    # Extract readable text from PDF
    reader = PdfReader(pdf_file)

    raw_text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            raw_text += page_text + "\n"

    return raw_text

raw_text = load_document(pdf_url)

print(f"Loaded {len(raw_text):,} characters")
print("-"*100)
print(raw_text[:500])

#### **Step 2. Chunking**

In [ ]:
CHUNK_SIZE = 500 # Maximum number of words per chunk
CHUNK_OVERLAP = 10 # Number of repeated words between chunks

# creates chunks of up to 300 words with 30 words of overlap
def create_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[dict]:

    clean_lines = []
    for line in text.splitlines(): # splitlines() separates the document wherever a line break occurs
        line = line.strip() # strip() removes spaces and tabs from the beginning and end of a line

        if line:
            clean_lines.append(line)

    words = " ".join(clean_lines).split()

    #print(words[:5])

    chunks = []
    for i in range(0, len(words), chunk_size):
        content = " ".join(words[i : i + chunk_size])
        chunks.append({
            "chunk_index": len(chunks),
            "content": content,
        })

    return chunks

chunks = create_chunks(raw_text)

print(f"Total chunks: {len(chunks)}")

for chunk in chunks[:3]:
  print("-" * 100)
  print(chunk)


#### **Step 3. Embeddings**

In [ ]:
from sentence_transformers import SentenceTransformer

# initializing the embedding model
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
chunk_texts = [chunk["content"] for chunk in chunks]

embeddings = embedder.encode(
    chunk_texts,
    convert_to_numpy=True
)

print("Total embeddings:", len(embeddings))
print(f"Shape: {embeddings.shape}")

In [ ]:
print(embeddings[0])

###### **Validating the chunks**

In [ ]:
import numpy as np

assert len(chunks) == len(embeddings)
embedding_dimension = embedder.get_sentence_embedding_dimension()

valid_data = []

for chunk, embedding in zip(chunks, embeddings):

    content = chunk["content"].strip()

    vector = np.asarray(embedding, dtype=np.float32)

    if not content:
        continue

    if vector.ndim != 1 or vector.size != embedding_dimension:
        raise ValueError(f"Invalid embedding dimension: {vector.shape}")

    if not np.isfinite(vector).all():
        raise ValueError("Embedding contains NaN or infinity")

    valid_data.append((chunk, vector))


print("Valid chunks:", len(valid_data))

##### **Create Qdrant collection**

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue)

# initializing the vectorDB client
client = QdrantClient(":memory:")

COLLECTION_NAME = "src_pdf"
DIM = embedder.get_sentence_embedding_dimension()

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=DIM,
        distance=Distance.COSINE,
    ),
)
print("Qdrant collection created successfully.")

#### **Step 4. Indexing**

Qdrant stores these as points.

A point consists of:
- An ID
- A vector (embedding)
- A payload (additional information, such as the original text)

In [ ]:
points = []

for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):

    # Skip empty text chunks
    if not chunk["content"].strip():
        print(f"Skipping empty chunk: {idx}")
        continue

    # Convert embedding to a Python list
    vector = embedding.tolist()

    # Skip empty embeddings
    if len(vector) == 0:
        print(f"Skipping empty embedding: {idx}")
        continue

    point = PointStruct(
        id=idx,
        vector=vector,
        payload={
            "content": chunk["content"]
        }
    )

    points.append(point)

print("Total points created:", len(points))

In [ ]:
# upsert = Inserts or updates points in Qdrant
result = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)
print(f"Status: {result.status}")

In [ ]:
info = client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")
print(f"Dimensions : {info.config.params.vectors.size}")

In [ ]:
stored_points = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[0],
    with_payload=True,
    with_vectors=True
)

for point in stored_points:
    print("ID:", point.id)
    print("Payload:", point.payload)
    print("Embedding:", point.vector)

#### **Step 5. Retrieval**
Converts the user's question into an embedding and retrieves the most similar chunks from Qdrant.

In [ ]:
#Embed the query and return the top-k most similar chunks.
def retriever(query: str, top_k: int = 5) -> list[dict]:

    if not query.strip():
        raise ValueError("Query cannot be empty")

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero")

    query_vector = embedder.encode(query).tolist()

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    results = []

    for hit in hits.points:
        result = {
            **hit.payload,
            "score": round(hit.score, 4)
        }
        results.append(result)

    return results

In [ ]:
results = retriever("What GPA do I need to maintain my scholarship?")

for result in results:
    #print("Chunk:", result["id"])
    print("Score:", result["score"])
    print("Content:", result["content"])
    print("-" * 50)


#### **Step 6. Context Augmentation**

In [ ]:
SYSTEM_PROMPT = """You are a helpful and professional Student Financial Aid and Scholarship Assistant for a college.

Your role is to answer student questions about financial aid, scholarships, grants, tuition fees, payment plans, refunds, emergency financial assistance, eligibility requirements, application procedures, deadlines, and college financial policies.

Follow these instructions carefully:

1. USE ONLY PROVIDED CONTEXT
- Answer the user's question using ONLY the information provided in the retrieved context.
- Do not use your own knowledge, assumptions, or external information.
- Never invent policies, eligibility criteria, deadlines, amounts, or application procedures.
- If the retrieved context does not contain enough information, clearly say that the information is not available in the provided policy
- Provide accurate, clear, and easy-to-understand answers.
- Use simple, student-friendly language

3. CITATIONS AND SOURCES
- Cite the relevant section name whenever you provide policy-related information.
- Use the section names exactly as they appear in the provided context whenever possible.
- Include citations in this format: [Source: Section Name].
- If multiple sections support your answer, cite each relevant section.
- Never invent section names or citations.


5. INSUFFICIENT OR MISSING INFORMATION
- If the answer cannot be found in the retrieved context, respond:
  "I’m sorry, but I could not find this information in the available financial aid and scholarship policy documents. Please contact the Financial Aid Office for clarification."
- If only part of the question can be answered, answer the supported part and clearly identify what information is missing.
- Do not fill gaps with assumptions.

6. RESPONSE FORMAT
- Start with a direct answer to the student's question.
- Use bullet points or numbered lists for multiple requirements or steps.
- Use headings when explaining complex policies.
- Keep answers concise but informative.
- Highlight important deadlines, amounts, and eligibility requirements when relevant.

7. PROFESSIONAL AND STUDENT-FRIENDLY TONE
- Be polite, supportive, patient, and professional.
- Do not judge students based on their financial circumstances or academic performance.
- If appropriate, guide students to contact the Financial Aid Office for questions requiring individual review.

8. DOCUMENT SUMMARIZATION
- When asked to summarize, provide the summary directly.
- Follow the requested word limit.
- Avoid unnecessary reasoning or explanations.
- Do not describe your summarization process.
- Include the most important policy information.
- Use only the retrieved context.
- Always provide a visible textual answer

Remember: Your responsibility is to provide reliable answers grounded strictly in the retrieved college financial aid and scholarship policy context. Accuracy is more important than providing an answer to every question.
"""

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [ ]:
from groq import Groq

# initialize Groq model
groq_client = Groq()
GROQ_MODEL  = "openai/gpt-oss-20b"

def rag_pipeline(query: str, top_k: int = 5):
    print("1. Starting retrieval...")

    # retrieve relevant chunks
    chunks = retriever(query, top_k=top_k)
    print("2. Retrieved chunks: ", len(chunks))

    if not chunks:
      return "No relevant content found in the document.", "", []

    context = build_context(chunks)

    print("3. Context length: ", len(context))
    print("4. Sending request to Groq...")

    user_message = f"Context:\n{context}\n\nQuestion: {query}"


    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.1,
    )

    # extract generated answer
    answer = response.choices[0].message.content

    if not answer or not answer.strip():
      print("Warning: Groq returned an empty response.")
      print("Finish reason:", response.choices[0].finish_reason)
      print("Usage:", response.usage)
      answer = "The model did not generate an answer. Please try again."

    return answer, context, chunks

In [ ]:
query = "summarize the document in 300 words"

answer, context, chunks = rag_pipeline(query, top_k=5)

print(f"\nQuestion: {query}\n")
print("Answer generated:")
print(answer)

print("\nRetrieved Sources:")

for i, chunk in enumerate(chunks, 1):

    print(f"\nSource [{i}]")
    print(chunk["content"])

    print(f"Similarity Score: {chunk['score']}")


In [ ]:
'''
Test Queires

1. What are the eligibility requirements for scholarships?
2. What are the main topics covered in this document?
3. Do I have to maintain any GPA in order to apply for a scholarship?
4. summarize the document in 100 words (no answer)
5. summarize the document in 300 words (generates answer)
'''
